# 🧪 DART로 삼성전자 기준값 만들기 — Student

주가 뒤의 재무 숫자를 직접 받고, 에이전트가 받은 값과 비교할 기준값을 만든다.

> **🖥️ 환경 설정**
>
> - **프로젝트 열기**: VS Code `File → Open Folder`에서 `FD_26`를 연다. 왼쪽에 `pyproject.toml`이 보여야 한다.

> - **터미널과 패키지**: 메뉴 `Terminal → New Terminal`에서 `uv --version`을 확인하고, 필요한 패키지가 없을 때만 `uv add pandas requests python-dotenv supabase`를 실행한다. 터미널은 명령어를 입력하는 창이다.

> - **커널**: 노트북 오른쪽 위 `Select Kernel`에서 `FD_26/.venv`를 선택한다. 실행 셀에 파이썬 경로가 보이면 준비가 된 것이다.
>
> - **의존성**: `pandas`·`requests`·`python-dotenv`·`supabase`가 없으면 `FD_26` 루트에서 `uv add pandas requests python-dotenv supabase`를 한 번 실행한다(이미 있으면 건너뜀).
>
> - **기본 런타임**: VS Code Copilot Chat(에이전트 모드)이 기본이다. Claude Code·Codex·Agy를 이미 쓰고 있다면 그걸 그대로 쓴다.
>
> - **데이터**: 이 노트북은 같은 폴더의 `data/`에 있는 실습 데이터를 사용한다.
>
> - **폴더**: Week04 폴더를 `FD_26/` 안에 둔다. `.env`는 `FD_26` 루트에서 찾는다.
>
> **⌨️ 단축키 안내**
>
> | 단축키 | 동작 |
> |--------|------|
> | `Shift + Enter` | 셀 실행 후 다음 셀 이동 |
> | `Esc → A` | 위에 셀 삽입 |
> | `Esc → B` | 아래에 셀 삽입 |
> | `Esc → DD` | 셀 삭제 |
> **🏠 수업 전 과제**
>
> - OpenDART 인증키를 발급받고 `FD_26` 루트의 `.env`에 `OPENDART_API_KEY`를 넣어 온다.
>
> - 준비하지 못했거나 API가 응답하지 않으면 아래 fixture로 계속 진행할 수 있다.
>
> - FD_26 루트에서 `uv add pandas requests python-dotenv supabase`를 한 번 실행해 온다(이미 있으면 건너뜀).


In [ ]:
# 🖥️ 환경 설정
# - 커널 선택 3단계: (1) 우측 상단 커널 클릭 → (2) FD_26/.venv 선택 → (3) 경로가 FD_26인지 확인한다.
# - VS Code에서 FD_26/.venv 커널을 선택했는지 먼저 확인한다 (실행·설치는 전부 FD_26에서).

import os
import sys

print(sys.executable)
# 예상 출력: (자기 프로젝트 경로)/FD_26/.venv/bin/python3


## 🔍 PER·PBR·ROE — 받은 숫자를 해석하는 세 개의 눈

[→ §100 강의노트](../notes/FD-04-100__why-and-what-to-load.md)

먼저 **예시(seed) 재무 숫자**로 ROE 공식에 손을 익힌다. 이 381,432·11.0%는 뒤 §300-2의 OpenDART 값(452,068·10.4%)과 **다르게 나오는 것이 정상**이다 — seed는 검증 기준이 아니라 실습용 예시 값이다.

계산 전에 자기자본 100원으로 얼마를 벌었는지 예측해 보자.

In [ ]:
# 📌 §100-1 🔍 PER·PBR·ROE — 받은 숫자를 해석하는 세 개의 눈
# 코드를 작성해주세요 👇

# 예상 출력:
# 예상 출력: ROE = 11.0%

## 🧪 첫 요청 — 삼성전자 재무제표 받아보기

[→ §300 강의노트](../notes/FD-04-300__opendart-hard-way.md)

이번에는 같은 질문을 OpenDART 창구에 직접 보낸다. 키는 환경변수에서만 읽고, 응답의 상태와 첫 계정을 확인한다.

키 문자열을 노트북에 직접 적거나 출력하지 않는 것이 첫 번째 검증이다.

## 🔐 인증키는 주소창에 넣지 않는다

주소창·브라우저 기록·화면 캡처에 인증키가 남지 않도록, 아래 `requests` 코드에서 환경변수 파일을 읽어 요청한다. 키 값은 셀에 붙여 넣거나 출력하지 않는다.

In [ ]:
# 📌 §300-1 🧪 첫 요청 — 삼성전자 재무제표 받아보기
# 코드를 작성해주세요 👇

# 예상 출력:
# 예상 출력([live] = 실제 API 호출 성공): [live] 000 유동자산 247,684,612,000,000
# 예상 출력([fixture] = 저장된 예시, API 호출 성공 아님): [fixture] 000 유동자산 247,684,612,000,000

## 🎯 여러 행 중에서 진짜 필요한 두 숫자만 골라내기

[→ §300 강의노트](../notes/FD-04-300__opendart-hard-way.md)

응답에서 연결(CFS) 기준의 당기순이익과 자본총계를 골라 억원으로 바꾼다. 먼저 전체 행 수와 계정 이름을 확인한 뒤 기준값을 만든다.

In [ ]:
# 📌 §300-2 🎯 여러 행 중에서 진짜 필요한 두 숫자만 골라내기
# 코드를 작성해주세요 👇
# ① 전체 조회: 먼저 len(data["list"])와 account_nm을 출력해 행의 규모를 본다.
# ② 눈으로 찾기: 당기순이익(손실)·자본총계가 CFS/OFS에 반복되는지 확인한다.
# ③ 조건으로 좁히기: fs_div == "CFS"와 정확한 account_nm을 함께 사용한다.
# ④ 쉼표 제거: thstrm_amount의 콤마를 지우고 int로 바꾼다.
# ⑤ 단위 맞추기: 원을 1e8로 나누어 억원으로 만든다.
# ⑥ 검산하기: 연결 당기순이익과 연결 자본총계를 각각 출력한다.

# 예상 출력:
# 예상 출력: 연결 당기순이익: 452,068억원 (45.2조원)
# 예상 출력: 연결 자본총계: 4,363,203억원 (436.3조원)

## 다음 단계

이제 45.2조 원이라는 기준값을 손에 쥐었다. 다음 노트북에서는 에이전트가 자연어로 받아온 값을 SQLite에 남기고, 같은 기준으로 적재와 지표를 검증한다.